![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 23 -- Lab 1: Scenes with a VGG Style CNN

Kaggle's Intel Image Classification set: 14,034 training photos of six kinds of scene (buildings, forest, glacier, mountain, sea, street), 3,000 labelled test photos and 7,301 unlabelled ones to predict, all 150 x 150. You cache the photos as tensors at 64 x 64 once, train Day 22's plain CNN and then the deck's VGG-style network (blocks with BatchNorm, global average pooling) with the same loop, compare their curves and parameter counts, test the better one once, and write predictions for the unlabelled folder.

**Your role:** Fill in the `# Your code here` cells in order. Run every cell, including the ones already written for you.

**Dataset:** Kaggle `puneet6060/intel-image-classification` (about 370 MB), downloaded by `kagglehub` in the setup cell: `seg_train` and `seg_test` with one folder per class, `seg_pred` with loose files. The setup cell resizes every photo to 64 x 64 and stores it as a `uint8` tensor (about 300 MB in memory), so each epoch runs from memory. Turn on a GPU in Colab (Runtime, Change runtime type, T4): on a CPU an epoch of the VGG-style network takes a few minutes.

---
# Setup

In [ ]:
# kagglehub, torch, torchvision, matplotlib, pandas and scikit-learn are preinstalled on Colab. Uncomment if an import fails.
# !pip install -q kagglehub torch torchvision matplotlib pandas scikit-learn

In [ ]:
import os, time
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

In [ ]:
path = kagglehub.dataset_download("puneet6060/intel-image-classification")
TRAIN, TEST, PRED = [os.path.join(path, f"seg_{s}", f"seg_{s}") for s in ("train", "test", "pred")]
to_u8 = transforms.Compose([transforms.Resize((64, 64)), transforms.PILToTensor()])   # (3, 64, 64) uint8, 0..255

def cache_folder(folder):
    """Decode every photo once, resized to 64 x 64, into one uint8 tensor (fixed work done once, Day 19)."""
    ds = datasets.ImageFolder(folder, transform=to_u8)
    xs, ys = [], []
    for xb, yb in DataLoader(ds, batch_size=256, num_workers=2):
        xs.append(xb); ys.append(yb)
    return torch.cat(xs), torch.cat(ys), ds.classes

t0 = time.time()
X_all, y_all, CLASSES = cache_folder(TRAIN)
X_test, y_test, _ = cache_folder(TEST)
pred_files = sorted(os.listdir(PRED))
X_pred = torch.stack([to_u8(Image.open(os.path.join(PRED, f)).convert("RGB")) for f in pred_files])
print(X_all.shape, X_all.dtype, X_test.shape, X_pred.shape, CLASSES, f"{time.time() - t0:.0f} s")

---
# Part 1 -- Look and split

The photos are `uint8` tensors in memory; the loop turns each batch into floats 0 to 1 with `.float() / 255` just before the model.

## Task 1: Look at the data

Print how many training photos each class has (`torch.bincount(y_all)` next to `CLASSES`). Then show 8 photos `X_all[i]` for `i in range(0, 14034, 1750)` with their class names (`permute(1, 2, 0)` for `imshow`; `uint8` images display as they are).

In [ ]:
# Your code here


## Task 2: Split and load

Wrap the training tensors in `TensorDataset(X_all, y_all)` and split it into 12,000 training and 2,034 validation photos with `random_split(..., [12000, 2034], generator=torch.Generator().manual_seed(0))` as `train_set, val_set`. Build `train_loader` (batch 128, `shuffle=True`), `val_loader` (batch 512) and `test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=512)`. Take one batch and print its shapes and dtype: `(128, 3, 64, 64)`, `torch.uint8`.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert len(train_set) == 12000 and tuple(xb.shape) == (128, 3, 64, 64) and xb.dtype == torch.uint8, 'split: TensorDataset, random_split 12000 / 2034, batches of 128 uint8 images'
print('Task 2 passed')

---
# Part 2 -- Two model classes

The *In Code: A Block Function and a Model Class* slide. Both models take `(B, 3, 64, 64)` floats and return 6 scores.

## Task 3: Day 22's plain CNN

Write `class PlainCNN(nn.Module)` with `n_classes=6`: `self.features` of three blocks `nn.Conv2d(c_in, c_out, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2)` for 3 to 16, 16 to 32, 32 to 64, and `self.classifier = nn.Linear(64 * 8 * 8, n_classes)`; `forward` runs the features, flattens with `flatten(1)` and applies the classifier. Create `plain = PlainCNN().to(device)`, run a float batch through it (`xb.to(device).float() / 255`) and print the output shape `(128, 6)` and the parameter count `n_plain` (expected 48,166).

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert tuple(out.shape) == (128, 6) and n_plain == 48166, 'PlainCNN: three conv-ReLU-pool blocks 3->16->32->64, then Linear(4096, 6)'
print('Task 3 passed')

## Task 4: The VGG-style network

Write `block(c_in, c_out)` returning `nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU())`, then `class VGGSmall(nn.Module)` with `n_classes=6`: `self.features` of three stages, each two blocks and a `nn.MaxPool2d(2)`, widths 3 to 32 to 32, 32 to 64 to 64, 64 to 128 to 128; `self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, n_classes))`; `forward` returns `self.head(self.features(x))`. Create `vgg = VGGSmall().to(device)`, print the output shape for the same batch and the parameter count `n_vgg` (expected 288,678). How many of those sit in the head?

Syntax hint (the shape of the call, not the answer):

```python
def block(c_in, c_out):
    return nn.Sequential(...)      # conv, BatchNorm2d, ReLU

class MyNet(nn.Module):
    def __init__(self, n_classes=6):
        super().__init__()
        self.features = nn.Sequential(block(...), block(...), nn.MaxPool2d(2), ...)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(..., n_classes))
    def forward(self, x):
        return self.head(self.features(x))
```

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert n_vgg == 288678 and n_head == 774, 'VGGSmall: 3 stages of 2 blocks (32, 64, 128) with BatchNorm2d, GAP head Linear(128, 6)'
print('Task 4 passed')

---
# Part 3 -- One epoch and one evaluation

The loop of Day 22, split into the two functions every competition notebook has: one pass over the training loader, one scoring pass over any loader. The epoch loop itself is written out in the next part.

## Task 5: train_one_epoch and evaluate

Write `train_one_epoch(model, loader, opt)`: `model.train()`, then for each batch `xb = xb.to(device).float() / 255`, `yb = yb.to(device)`, the four moves with `loss_fn = nn.CrossEntropyLoss()`, summing `loss.item() * len(yb)`; return the mean loss over the loader. Write `evaluate(model, loader)`: `model.eval()`, inside `torch.no_grad()` the same conversion, summing the loss and the number of correct `argmax` predictions; return `(mean_loss, accuracy)`. Test `evaluate(vgg, val_loader)` on the untrained network: loss about `ln(6) = 1.79`, accuracy about 0.17.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert 1.5 < val_loss0 < 2.2 and val_acc0 < 0.5, 'evaluate: eval mode, no_grad, floats 0..1, mean loss and accuracy over the loader'
print('Task 5 passed')

---
# Part 4 -- Train both, compare

The *Where Depth Breaks* and *The Recipe* slides: same data, same loop, same number of epochs; only the model class changes.

## Task 6: Train the plain CNN

Set `EPOCHS = 6` and `opt = torch.optim.Adam(plain.parameters(), lr=1e-3)`. Write the epoch loop: for each epoch call `train_one_epoch(plain, train_loader, opt)`, then `evaluate(plain, val_loader)`, append `(train_loss, val_loss, val_acc)` to `hist_plain`, save `plain.state_dict()` to `best_plain.pt` when the validation accuracy improves, and print the three numbers. Expected: validation accuracy around 0.70 after 6 epochs.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert len(hist_plain) == 6 and best_plain > 0.6 and os.path.exists('best_plain.pt'), 'plain loop: 6 epochs, train_one_epoch then evaluate, checkpoint on the best validation accuracy'
print('Task 6 passed')

## Task 7: Train the VGG-style network

The same epoch loop for `vgg` (a fresh `torch.optim.Adam(vgg.parameters(), lr=1e-3)`, 6 epochs, history `hist_vgg`, checkpoint `best_vgg.pt`). Then plot the validation accuracy of both runs against the epoch on one axis (markers, a legend naming the model and its parameter count) and print the best validation accuracy of each. Expected: the VGG-style network ahead by about ten points (around 0.80).

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert len(hist_vgg) == 6 and best_vgg > 0.65, 'VGG loop: same six epochs; BatchNorm blocks and the GAP head should pass 0.65'
print('Task 7 passed')

---
# Part 5 -- Test once, predict the unlabelled folder

An Olympiad hand-in: the better model's best checkpoint, scored once on `seg_test`, then predictions for every file in `seg_pred`.

## Task 8: Test the best checkpoint

Create `best = VGGSmall().to(device)`, load `best_vgg.pt` (`map_location=device`), and evaluate it on `test_loader` as `test_loss, test_acc`. Then collect the predictions on the test set in order (`argmax`, `.cpu()`) into `pred_test` and print `confusion_matrix(y_test.numpy(), pred_test)` as a DataFrame with the class names. Which two scenes are confused most?

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert pred_test.shape == (3000,) and test_acc > 0.65, 'test: load best_vgg.pt, evaluate on test_loader, one prediction per test photo in order'
print('Task 8 passed')

## Task 9: Predict the unlabelled folder and write the submission

Run `X_pred` through `best` in batches of 512 (`torch.no_grad()`, floats 0 to 1, `argmax`, `.cpu()`) into `pred_labels`, build `pd.DataFrame({"image": pred_files, "label": [CLASSES[p] for p in pred_labels]})`, save it as `submission.csv` without the index, read it back and print its shape and first 5 rows (7,301 rows). Then show the first 8 predicted photos with their predicted labels.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
check = pd.read_csv('submission.csv')
assert check.shape == (7301, 2) and set(check['label'].unique()) <= set(CLASSES), 'submission.csv: one row per file of seg_pred, columns image and label'
print('Task 9 passed')

---
## Task 10: Reflect

VGGSmall has six times the parameters of PlainCNN and only 774 of them in its head. Where are the rest, and which two ingredients of the deck's recipe made it train better in the same six epochs? Each epoch took a few seconds because the photos were cached as tensors. What would change if the loader opened the JPEG files every epoch, and why does Lab 2 do exactly that anyway? The test accuracy was measured once on 3,000 labelled photos you never used for a decision. If you had picked the epoch on `seg_test` instead of `val_loader`, what would the number mean?

*Your answers here*

---
## Conclusion

- **Data as tensors**: a Kaggle folder cached once at 64 x 64 as `uint8`, converted to floats in the loop, split with `random_split`.
- **Two model classes**: Day 22's plain CNN and the deck's VGG-style network built from a `block` function with BatchNorm and a global-average-pooling head, checked by shape and parameter count.
- **The loop**: `train_one_epoch` and `evaluate` written once, the epoch loop written out per model with a checkpoint on validation; the deeper, normalised network won the comparison.
- **The hand-in**: the best checkpoint tested once, a confusion matrix, and a submission file for every unlabelled photo.

---

Made By **Sattam Altwaim**